# Notebook 1: OpenAI Agents SDK — Basics

**What you'll learn:** What is the Agents SDK, why it exists, its core primitives, and your first agent in 5 lines.

---

## What is an AI Agent?

An **AI Agent** is NOT just a chatbot. It is:

```
Agent = LLM + Instructions + Tools + Decision-Making Loop
```

A chatbot answers questions. An agent **takes actions**, **makes decisions**, and **completes tasks autonomously**.

### Real-World Analogy
- **Chatbot** = A librarian who answers your questions
- **Agent** = A personal assistant who books flights, sends emails, checks weather, and plans your day

---

## Why OpenAI Agents SDK?

| Feature | Raw LLM API | LangChain | **Agents SDK** |
|---|---|---|---|
| Complexity | Manual everything | Heavy abstractions | **Minimal primitives** |
| Learning Curve | Low (but manual) | Steep | **Gentle** |
| Production Ready | DIY | Yes | **Yes** |
| Tool Calling | Manual JSON | Chain-based | **Decorator-based** |
| Multi-Agent | Build yourself | Complex graphs | **Built-in handoffs** |
| Provider Lock-in | OpenAI only | Any | **Any (100+ LLMs)** |

The SDK evolved from OpenAI's experimental **Swarm** project into a production-grade framework.

---

## The 4 Core Primitives

The entire SDK is built on just **4 concepts**:

```
1. Agent        → The brain (LLM + instructions + tools)
2. Runner       → The executor (runs the agent loop)
3. Tools        → The hands (functions the agent can call)
4. Handoffs     → The delegation (agent-to-agent transfer)
```

That's it. No graphs, no nodes, no edges, no state machines.

## Installation

In [1]:
# Install the SDK
# uv add openai-agents

## Local Model Setup (Ollama)

We use a local LLM via Ollama. No API key needed, fully private.

In [2]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

# Disable tracing — not needed for local models
set_tracing_disabled(True)

# Ollama client pointing to local server
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Local model wrapper
local_model = OpenAIChatCompletionsModel(
    model="llama3.2:latest",
    openai_client=ollama_client
)

print("Local model ready: llama3.2:latest")

Local model ready: llama3.2:latest


---

## Your First Agent — Hello World

In [3]:
# Step 1: Create an Agent with local model
agent = Agent(
    name="Greeter",
    instructions="You are a friendly assistant. Be concise.",
    model=local_model
)

# Step 2: Run it (async — required in Jupyter)
result = await Runner.run(agent, "Hello! What can you do?")

# Step 3: Get the output
print(result.final_output)

I'm here to help with answers, information, and tasks such as:

- Answering questions on various topics
- Generating text and summaries
- Translating languages
- Suggesting recipes and creative content ideas
- Providing definitions and explanations
- Assisting with writing and proofreading
- Chatting about hobbies and interests

What's on your mind?


### Breaking it down:

```python
Agent(name, instructions, model=local_model)  # WHO the agent is
await Runner.run(agent, input)                 # RUN the agent with user input  
result.final_output                            # GET what the agent produced
```

**That's the entire mental model.** Everything else builds on this.

---

## Understanding the Result Object

The `RunResult` object contains much more than just the output:

In [4]:
agent = Agent(
    name="Analyst",
    instructions="You analyze topics briefly.",
    model=local_model
)

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

Final Output:  Python's popularity in the AI field can be attributed to several factors:

1. **Easy-to-learn syntax...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


---

## The Agent Loop — How It Actually Works

When you call `Runner.run()`, here's what happens internally:

```
┌─────────────────────────────────────────────┐
│           Runner.run(agent, input)          │
│                                             │
│  ┌──────────────────────────────────────┐   │
│  │ 1. Send input + instructions to LLM  │   │
│  │ 2. LLM responds                      │   │
│  │ 3. Check response:                   │   │
│  │    ├─ Final output? → RETURN result  │   │
│  │    ├─ Tool call?    → Execute tool,  │   │
│  │    │                  loop back to 1 │   │
│  │    └─ Handoff?      → Switch agent,  │   │
│  │                       loop back to 1 │   │
│  └──────────────────────────────────────┘   │
│                                             │
│  This loop IS the "agentic" behavior.       │
│  The LLM DECIDES what to do next.           │
└─────────────────────────────────────────────┘
```

**Key insight:** The agent doesn't just answer — it enters a **reasoning loop** where it can call tools, get results, reason again, and repeat until done.

---

## Multi-Turn Conversations

Agents can maintain conversation context across turns using `to_input_list()`:

In [5]:
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: **Lists in Python**

A list in Python is an ordered collection of items that can be of any data type, including strings, integers, floats, and other lists. Lists are denoted by square brackets `[]` and are mutable, meaning they can be modified after creation.

**Key Features**

*   Ordered: Lists maintain the order in which elements were added.
*   Mutable: Elements can be added or removed from a list.
*   Indexable: Elements can be accessed using their index (position in the list).

**Basic Operations**
-------------------

Here's an example of creating and manipulating lists:

```python
# Create a new list
fruits = ['apple', 'banana', 'cherry']

# Access an element by its index
print(fruits[0])  # Output: apple

# Modify an element
fruits[0] = 'mango'
print(fruits)  # Output: ['mango', 'banana', 'cherry']

# Add an element to the end of the list
fruits.append('orange')
print(fruits)  # Output: ['mango', 'banana', 'cherry', 'orange']
```

**Common List Methods**
--------------

### Key Pattern:
```python
# This is how you maintain conversation memory:
new_input = previous_result.to_input_list() + [{"role": "user", "content": "next message"}]
result = await Runner.run(agent, new_input)
```

---

## 3 Ways to Run an Agent

| Method | Use Case | Async? |
|---|---|---|
| `Runner.run()` | Jupyter, async apps, FastAPI | Yes |
| `Runner.run_sync()` | Scripts, CLI tools | No |
| `Runner.run_streamed()` | Real-time UIs, streaming | Yes |

In [6]:
agent = Agent(
    name="Streamer",
    instructions="Explain briefly.",
    model=local_model
)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

Machine learning (ML) is a subset of artificial intelligence that enables computers to learn and improve their performance on specific tasks without being explicitly programmed, by analyzing data and detecting patterns. Through algorithms and models, machines can make accurate predictions, classify objects, generate insights, and automate processes, all with minimal human intervention.

--- Final: Machine learning (ML) is a subset of artificial intelligence that enables computers to learn and improve their performance on specific tasks without being explicitly programmed, by analyzing data and detecting patterns. Through algorithms and models, machines can make accurate predictions, classify objects, generate insights, and automate processes, all with minimal human intervention.


---

## Exercise: Build Your First Agent

Create an agent that acts as a **Lahore travel guide**. Ask it 3 questions in a multi-turn conversation.

In [7]:
# EXERCISE SOLUTION: Lahore Travel Guide Agent

lahore_guide = Agent(
    name="Lahore Guide",
    instructions=(
        "You are an expert Lahore travel guide. You know everything about Lahore's "
        "history, food, culture, landmarks, and local tips. Be friendly and concise."
    ),
    model=local_model
)

# Turn 1
result = await Runner.run(lahore_guide, "What are the top 3 places to visit in Lahore?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry history
new_input = result.to_input_list() + [{"role": "user", "content": "What food should I try there?"}]
result = await Runner.run(lahore_guide, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "Best time of year to visit?"}]
result = await Runner.run(lahore_guide, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: Lahore is a treasure trove of history, architecture, and cultural heritage. Here are my top 3 recommendations for visitors:

1. **Mughal Mosque**: Also known as Badshahi Mosque, this stunning mosque was built by Emperor Aurangzeb in 1673 AD. Its majestic minarets, gleaming tile work, and intricate carvings will leave you awestruck.

2. **Wazir Khan Mosque**: This beautiful Mughal-era mosque (1634) is famous for its white marble pavement adorned with calligraphy and intricate tile work. During Ramadan, the mosque comes alive with moonlight sermons.

3. **Shahi Masjid** (Lahore Fort): The imposing citadel has served as a palace, fort, governorate, and royal residence throughout history. This stunning complex features two iconic mosques, Shri Mahal, Nihari Mahal, the Wazir Bagh, and much more.

These three attractions showcase Lahore's rich Mughal past, architectural prowess, and tranquil beauty.

Turn 2: Lahore is a foodie's paradise! Here are some iconic dishes you absolutely ca

---

## Summary — What You Learned

| Concept | Code |
|---|---|
| Create an agent | `Agent(name, instructions, model=local_model)` |
| Run async | `await Runner.run(agent, input)` |
| Run sync | `Runner.run_sync(agent, input)` |
| Run streaming | `Runner.run_streamed(agent, input)` |
| Get output | `result.final_output` |
| Continue conversation | `result.to_input_list() + [new_msg]` |

**Next:** Notebook 2 — Creating Proper Agents with instructions, output types, and model settings.